# ATML PA3 · Task 1 · Step 2: Full fine-tuning with L2-SP

Runs end-to-end with **Run All** / **Save Version** on Kaggle (GPU **T4 x2** recommended, Internet ON).

What it does:
1. clones the repo and installs dependencies
2. `task1.prepare`: same data / probe set / prompts as every Task 1 run (data hash is checked)
3. `task1.import_prior`: copies results of earlier Task 1 notebooks attached under **Add Input**
   (attach the **Full FT notebook output** → shared baseline + Full FT results, so this report shows
   changes from baseline and the Full FT vs. L2-SP comparison). If no baseline is found and a second
   GPU exists, the baseline is evaluated on GPU 1 in parallel instead.
4. `task1.train` on **GPU 0**: Full FT + L2-SP, lr 2e-5, λ = 1e-3, Ω = ½‖θ−θ0‖² over all unique parameters. Logs Ω, λΩ and SFT+λΩ every update plus reference-storage/transfer overhead. Evaluated at the end only.
5. `task1.evaluate`: **final** model
6. `task1.diagnostics`: ρ_k, η, R(z) on `down_proj` (layers 0, 13, 27)
7. `task1.report`: tables, figures, qualitative sheets → **`/kaggle/working/task1_l2sp_report.zip`**

Training uses one GPU, as in every Task 1 run, so training time and memory are comparable.

If the session restarts, re-running resumes training from the latest recovery checkpoint and skips
evaluations that already finished. If a saved version failed **after** training, attach its output
under **Add Input** and run again: the FP16 exports in `task1_artifacts/exports/` are imported,
training is skipped and only the missing evaluations run.

In [ ]:
# ---- settings ---------------------------------------------------------------------------
REPO_URL   = "https://github.com/alihasnaink/finetune-ground-scale.git"
BRANCH     = "main"
CONFIG     = "task1/configs/l2sp.yaml"
RUN        = "l2sp"
SMOKE_TEST = False   # True = tiny end-to-end pipeline check (~15-20 min), separate output folder
PRIOR_SEARCH = "/kaggle/input"   # where attached outputs of earlier Task 1 notebooks are mounted

REPO_DIR = "/kaggle/working/repo"
if SMOKE_TEST:
    OUT_ROOT, SCRATCH = "/kaggle/working/task1_smoke", "/tmp/atml_task1_smoke"
    ARTIFACTS = "/kaggle/working/task1_smoke_artifacts"
    OVERRIDES = ["data.num_examples=256", "train.num_updates=8", "train.warmup_updates=2",
                 "train.checkpoint_updates=[4,8]", "eval.limit=16"]
else:
    OUT_ROOT, SCRATCH = "/kaggle/working/task1", "/tmp/atml_task1"
    ARTIFACTS = "/kaggle/working/task1_artifacts"   # FP16 exports + FP32 deltas (kept, not zipped)
    OVERRIDES = []
ZIP_PATH = f"/kaggle/working/task1_{RUN}{'_smoke' if SMOKE_TEST else ''}_report.zip"

In [ ]:
!nvidia-smi
import os, subprocess
if os.path.isdir(f"{REPO_DIR}/.git"):
    subprocess.run(["git", "-C", REPO_DIR, "fetch", "origin", BRANCH], check=True)
    subprocess.run(["git", "-C", REPO_DIR, "checkout", BRANCH], check=True)
    subprocess.run(["git", "-C", REPO_DIR, "reset", "--hard", f"origin/{BRANCH}"], check=True)
else:
    subprocess.run(["git", "clone", "--branch", BRANCH, REPO_URL, REPO_DIR], check=True)
print(subprocess.check_output(["git", "-C", REPO_DIR, "log", "-1", "--oneline"], text=True))

In [ ]:
!pip install -q -r {REPO_DIR}/requirements-task1.txt
import torch, transformers, lm_eval
print("torch", torch.__version__, "| transformers", transformers.__version__, "| lm_eval", lm_eval.__version__)
N_GPU = torch.cuda.device_count()
print("GPUs:", [torch.cuda.get_device_name(i) for i in range(N_GPU)])
assert N_GPU >= 1, "enable a GPU accelerator (T4 x2 recommended)"

In [ ]:
import sys, time
ENV = os.environ | {"T1_OUT_ROOT": OUT_ROOT, "T1_SCRATCH_ROOT": SCRATCH, "T1_ARTIFACTS_ROOT": ARTIFACTS,
                    "PYTHONUNBUFFERED": "1", "TOKENIZERS_PARALLELISM": "false",
                    "PYTORCH_CUDA_ALLOC_CONF": "expandable_segments:True",
                    "TQDM_MININTERVAL": "30", "HF_HUB_DISABLE_PROGRESS_BARS": "1"}

def _cmd(module, args, config=None):
    cmd = [sys.executable, "-m", f"task1.{module}", "--config", config or CONFIG, *args]
    return cmd + (["--set", *OVERRIDES] if OVERRIDES else [])

def run(module, *args, gpu="0", config=None):
    """Run a task1 script in the foreground, streaming its output; raise on failure."""
    t0 = time.time()
    p = subprocess.Popen(_cmd(module, args, config), cwd=REPO_DIR, env=ENV | {"CUDA_VISIBLE_DEVICES": gpu},
                         stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    for line in p.stdout:
        print(line, end="")
    if p.wait() != 0:
        raise RuntimeError(f"task1.{module} failed (exit {p.returncode})")
    print(f"--- task1.{module} finished in {(time.time() - t0) / 60:.1f} min")

def run_bg(module, *args, gpu="1", logfile, config=None):
    """Start a task1 script in the background on another GPU, logging to a file."""
    f = open(logfile, "w")
    return subprocess.Popen(_cmd(module, args, config), cwd=REPO_DIR, env=ENV | {"CUDA_VISIBLE_DEVICES": gpu},
                            stdout=f, stderr=subprocess.STDOUT, text=True)

os.makedirs(OUT_ROOT, exist_ok=True)

In [ ]:
# 1) data, probe set, qualitative prompts, model + eval-dataset downloads
run("prepare")

In [ ]:
# 2) import earlier Task 1 results (baseline, Full FT, ...) from attached notebook outputs;
#    if the baseline is still missing, evaluate it on GPU 1 while training runs on GPU 0
run("import_prior", "--search", PRIOR_SEARCH)
EVAL_LOG = f"{OUT_ROOT}/eval_worker_gpu1.log"
worker = None
have_baseline = os.path.exists(f"{OUT_ROOT}/eval/baseline/metrics.json")
if not have_baseline and N_GPU >= 2:
    worker = run_bg("evaluate", "--conditions", "baseline", gpu="1", logfile=EVAL_LOG)
    print("baseline not imported -> evaluating it on GPU 1, log:", EVAL_LOG)
elif not have_baseline:
    print("baseline not imported and only one GPU -> it will be evaluated after training")
else:
    print("baseline imported")

In [ ]:
# 3) training on GPU 0 (resumes automatically from the latest recovery checkpoint)
run("train", gpu="0")

In [ ]:
# 4) final evaluation on GPU 0 (plus anything the GPU-1 worker could not do)
conds = [f"{RUN}_final"] if worker is not None else [f"{RUN}_final"]
if worker is None and not os.path.exists(f"{OUT_ROOT}/eval/baseline/metrics.json"):
    conds = ["baseline"] + conds
run("evaluate", "--conditions", *conds, gpu="0")

In [ ]:
# 5) wait for the GPU-1 worker (if any) and show its log
if worker is not None:
    while worker.poll() is None:
        time.sleep(30)
    print(open(EVAL_LOG).read()[-6000:])
    assert worker.returncode == 0, f"eval worker failed (exit {worker.returncode}); see {EVAL_LOG}"

In [ ]:
# 6) weight-space and activation diagnostics
run("diagnostics", gpu="0")

In [ ]:
# 7) tables, figures, qualitative sheets and the downloadable zip
run("report", "--zip", ZIP_PATH)

In [ ]:
# quick look at the headline tables
from pathlib import Path
from IPython.display import Markdown, display, FileLink, Image
rep = Path(OUT_ROOT) / "report"
for t in ["benchmarks_primary", "gsm8k_behavior", "resources", "displacement", "l2sp_objective",
          "adapters", "adapter_subspace_by_module", "diagnostics_weight_space",
          "diagnostics_R_math_vs_nonmath"]:
    f = rep / "tables" / f"{t}.md"
    if f.exists():
        display(Markdown(f.read_text()))
for f in sorted((rep / "figures").glob("*.png")):
    display(Image(filename=str(f)))
display(FileLink(os.path.relpath(ZIP_PATH, "/kaggle/working")))

In [ ]:
# optional: free scratch space (recovery checkpoints are not part of the output)
# import shutil; shutil.rmtree(SCRATCH, ignore_errors=True)
!du -sh {OUT_ROOT} {ARTIFACTS} {ZIP_PATH} {SCRATCH} 2>/dev/null